# PHIStruct: Improving phage-host interaction prediction at low sequence similarity settings using structure-aware protein embeddings

<b>Mark Edward M. Gonzales<sup>1, 2</sup>, Jennifer C. Ureta<sup>1, 2</sup> & Anish M.S. Shrestha<sup>1, 2</sup></b>

<sup>1</sup> Bioinformatics Lab, Advanced Research Institute for Informatics, Computing and Networking, De La Salle University, Manila 1004 <br>
<sup>2</sup> Department of Software Technology, College of Computer Studies, De La Salle University, Manila 1004

✉️ gonzales.markedward@gmail.com, jennifer.ureta@gmail.com, anish.shrestha@dlsu.edu.ph

<hr>

# 💡 Prerequisites

### Option 1: Download the prerequisite files
1. Download `consolidated.tar.gz` from this [link](https://drive.google.com/file/d/1yQSXwlb37dm2ZLXGJHdIM5vmrzwPAwvI/view?usp=sharing), and unzip it. This should result in a folder named `consolidated`. <br> Technically, this notebook only needs `consolidated/rbp_embeddings_pst_relaxed_r3.csv`.
1. Create a folder named `inphared` inside `data`, and save the extracted `consolidated` folder inside `data/inphared`. 
1. Download `fasta.tar.gz` from this [link](https://drive.google.com/file/d/1NMFR3JrrrCHLoCMQp2nia4dgtcXs5x05/view?usp=sharing), and unzip it. This should result in a folder named `fasta`. <br> Technically, this notebook only needs the `.clstr` files inside `fasta`.
1. Save the extracted `fasta` folder inside `data/inphared`.

### Option 2: Generate the prerequisite files yourself
1. If you have run `3.2. Data Consolidation (PST).ipynb`, then `data/inphared/consolidated` should have already been populated with the prerequisite files.
1. Consolidate the sequences of the proteins with predicted structures into a single FASTA file. <br>
   For reproducibility, we provide our consolidated FASTA file [here](https://drive.google.com/file/d/1LTZte1f4lreQ5MXWeM-y2Mtp9z96pXS7/view?usp=sharing).
1. Generate the protein clusters by running CD-HIT on this FASTA file at a sequence similarity threshold of 100%, following the instructions [here](https://github.com/weizhongli/cdhit). 
1. Rename the resulting `.clstr` file to `complete-struct-100.fasta.clstr` and the resulting FASTA file (containing only the representative sequences) to `complete-struct-100.fasta`. 
1. Generate `complete-struct-80.fasta.clstr`, `complete-struct-60.fasta.clstr`, and `complete-struct-40.fasta.clstr` by running CD-HIT on `complete-struct-100.fasta` at sequence similarity thresholds of 80%, 60%, and 40%, respectively.
1. Create a folder named `fasta` inside `data/inphared`, and save the four `.clstr` files inside `data/inphared/fasta`.

### Resulting folder structure

`experiments` (parent folder of this notebook) <br> 
↳ `data` <br>
&nbsp; &nbsp;↳ `inphared` <br>
&nbsp; &nbsp;&nbsp; &nbsp; ↳ `consolidated` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `rbp_embeddings_pst_relaxed_r3.csv` <br>
&nbsp; &nbsp;&nbsp; &nbsp; ↳ `fasta` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-100.fasta.clstr` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-80.fasta.clstr` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-40.fasta.clstr` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-60.fasta.clstr` <br>
↳ `5.2. Benchmarking - Classifier Building & Evaluation (PST).ipynb` (this notebook) <br>

<hr>

# 📁 Output files

The output files (i.e., the results of evaluating the model's performance) &mdash; which are saved in `temp/results` &mdash; are already included when the repository was cloned. <br>

<hr>

# Part I: Preliminaries

Import the necessary libraries and modules.

In [1]:
import math
import pickle
import os
import warnings

import pandas as pd
import numpy as np
import sklearn

import ConstantsUtil
import ClassificationUtil

%load_ext autoreload
%autoreload 2

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", 50)

pd.options.mode.chained_assignment = None

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=sklearn.exceptions.UndefinedMetricWarning
    )

In [3]:
constants = ConstantsUtil.ConstantsUtil()
util = ClassificationUtil.ClassificationUtil()

<hr>

# Part II: Classifier Building and Evaluation

Train a multilayer perceptron, and evaluate its performance at different train-versus-test similarity and confidence thresholds.

In [4]:
models = list(constants.PST_PLM.keys())

for similarity in range(100, 39, -20):
    for model in models:
        model = model.lower()
        df, df_all, protein_clusters = util.filter_proteins_based_on_struct_and_seq_sim(
            f"{constants.INPHARED}/{constants.CONSOLIDATED}/rbp_embeddings_{model}.csv",
            f"{constants.INPHARED}/{constants.CONSOLIDATED}/rbp_embeddings_pst_relaxed_r3.csv",
            f"{constants.INPHARED}/{constants.FASTA}/complete-struct-{similarity}.fasta.clstr",
        )

        include_proteins_in_cluster = True
        if similarity == 100:
            include_proteins_in_cluster = False

        print(f"*** {model}, similarity = {similarity}% ***")
        util.classify(
            df,
            model + "-mlp-eskapee-smotetomek",
            similarity,
            genus=[
                "enterococcus",
                "staphylococcus",
                "klebsiella",
                "acinetobacter",
                "pseudomonas",
                "enterobacter",
                "escherichia",
            ],
            feature_columns=[f"s{i}" for i in range(1, 1281)],
            include_proteins_in_cluster=include_proteins_in_cluster,
            rbp_embeddings_all=df_all,
            protein_clusters=protein_clusters,
            undersample_others=True,
            oversample_technique="SMOTETomek",
            model="MLP",
        )

*** pst_relaxed_r3, similarity = 100% ***
Constructing training and test sets...
Training set shape: (16924, 1280)
Test set shape: (2340, 1280)
Training the model...
Saving evaluation results...
Confidence threshold k: 0.0%
                precision    recall  f1-score   support

 acinetobacter     0.8990    0.8018    0.8476       111
  enterobacter     0.3094    0.4828    0.3771       116
  enterococcus     0.7833    0.9216    0.8468        51
   escherichia     0.8740    0.8269    0.8498      1040
    klebsiella     0.8256    0.8829    0.8532       461
        others     0.0000    0.0000    0.0000        51
   pseudomonas     0.9066    0.9322    0.9192       354
staphylococcus     0.9308    0.9487    0.9397       156

      accuracy                         0.8278      2340
     macro avg     0.6911    0.7246    0.7042      2340
  weighted avg     0.8253    0.8278    0.8249      2340

Confidence threshold k: 10.0%
                precision    recall  f1-score   support

 acinetobacter

Confidence threshold k: 10.0%
                precision    recall  f1-score   support

 acinetobacter     0.3667    0.7064    0.4828       109
  enterobacter     0.1814    0.2575    0.2129       167
  enterococcus     0.5909    0.6500    0.6190        60
   escherichia     0.7996    0.6205    0.6988      1344
    klebsiella     0.5779    0.6369    0.6060       559
        others     0.0308    0.0333    0.0320        60
   pseudomonas     0.5609    0.6695    0.6104       351
staphylococcus     0.8675    0.8372    0.8521       172

      accuracy                         0.6130      2822
     macro avg     0.4970    0.5514    0.5142      2822
  weighted avg     0.6561    0.6130    0.6258      2822

Confidence threshold k: 20.0%
                precision    recall  f1-score   support

 acinetobacter     0.3793    0.7064    0.4936       109
  enterobacter     0.1830    0.2455    0.2097       167
  enterococcus     0.5909    0.6500    0.6190        60
   escherichia     0.7988    0.6057    0

Confidence threshold k: 20.0%
                precision    recall  f1-score   support

 acinetobacter     0.4198    0.7355    0.5345       121
  enterobacter     0.2067    0.2194    0.2129       196
  enterococcus     0.6333    0.7755    0.6972        49
   escherichia     0.7141    0.7074    0.7107      1589
    klebsiella     0.5653    0.3605    0.4403       588
        others     0.0300    0.1224    0.0482        49
   pseudomonas     0.7056    0.6287    0.6649       404
staphylococcus     0.7683    0.8025    0.7850       157

      accuracy                         0.6001      3153
     macro avg     0.5054    0.5440    0.5117      3153
  weighted avg     0.6332    0.6001    0.6099      3153

Confidence threshold k: 30.0%
                precision    recall  f1-score   support

 acinetobacter     0.4472    0.7355    0.5562       121
  enterobacter     0.2124    0.2092    0.2108       196
  enterococcus     0.6552    0.7755    0.7103        49
   escherichia     0.7223    0.6923    0

Confidence threshold k: 30.0%
                precision    recall  f1-score   support

 acinetobacter     0.2414    0.6292    0.3489        89
  enterobacter     0.2278    0.1667    0.1925       216
  enterococcus     0.4384    0.7805    0.5614        41
   escherichia     0.8026    0.5206    0.6315      1796
    klebsiella     0.3752    0.5446    0.4443       527
        others     0.0274    0.1951    0.0480        41
   pseudomonas     0.4973    0.5981    0.5431       311
staphylococcus     0.8472    0.6703    0.7485       182

      accuracy                         0.5189      3203
     macro avg     0.4322    0.5131    0.4398      3203
  weighted avg     0.6362    0.5189    0.5530      3203

Confidence threshold k: 40.0%
                precision    recall  f1-score   support

 acinetobacter     0.2478    0.6292    0.3556        89
  enterobacter     0.2518    0.1620    0.1972       216
  enterococcus     0.4507    0.7805    0.5714        41
   escherichia     0.8051    0.5084    0